# 🚗 EquiTraffic-GPT: Graph WaveNet (GWNet) Google Colab Training Pipeline

This notebook provides a **standalone, GPU-accelerated Google Colab environment** for training Graph WaveNet GNN models on **METR-LA (207 nodes)** and **San Diego SD400 (716 nodes)** highway sensor networks.

### 💡 Features:
1. **Universal PeMS Dataset Integration**: Supports METR-LA, SD400, PeMS04, PeMS08, PeMS-BAY, PeMS03, PeMS07.
2. **PyTorch 2.x FlashAttention & GCN Chebyshev Einsum**: High-performance spatial-temporal graph neural convolutions.
3. **Physics-Informed Custom Loss**: Bottleneck severity penalty ($lpha, eta$) from `model_config.yaml`.
4. **Automatic Model Checkpoint & Metadata Export**: Saves `.pt` and `.tar` state dict checkpoints directly to Google Drive or local workspace.

## 🛠️ Step 1: Environment Setup & Hardware Check

In [ ]:
# Check GPU Availability
!nvidia-smi

# Install Dependencies
!pip install -q torch numpy pandas pyyaml scipy matplotlib tqdm

## 📂 Step 2: Directory Structure & Dataset Loader Verification

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import yaml

# Set up paths
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
CODE_DIR = os.path.join(BASE_DIR, 'code')
CKPT_DIR = os.path.join(BASE_DIR, 'checkpoints')

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f'[+] Workspace Root: {BASE_DIR}')
print(f'[+] Data Directory: {DATA_DIR}')

# Load Config
config_path = os.path.join(CODE_DIR, 'model_config.yaml')
with open(config_path, 'r') as f:
    model_config = yaml.safe_load(f)
print('[+] Model Config Loaded Successfully:', model_config['graph_wavenet_gnn']['model_name'])

## 🧠 Step 3: Graph WaveNet (GWNet) Neural Architecture

In [ ]:
import torch
from gwnet_model import GraphWaveNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] PyTorch Active Compute Device: {device}')

# Instantiate Model for METR-LA (207 Nodes)
model_metr_la = GraphWaveNet(
    num_nodes=207,
    in_dim=3,
    out_dim=12,
    residual_channels=32,
    dilation_channels=32,
    skip_channels=256,
    end_channels=512
).to(device)

print('[+] METR-LA Model Built. Total Parameters:', sum(p.numel() for p in model_metr_la.parameters()))

## 🎯 Step 4: Physics-Informed Custom Loss Function

In [ ]:
from gwnet_loss import CustomPhysicsLoss

criterion = CustomPhysicsLoss(
    alpha=model_config['graph_wavenet_gnn']['loss']['alpha'],
    beta=model_config['graph_wavenet_gnn']['loss']['beta'],
    speed_threshold_norm=model_config['graph_wavenet_gnn']['loss']['default_speed_threshold_norm']
)
print('[+] Custom Physics Loss Function Initialized with Alpha=', criterion.alpha, 'Beta=', criterion.beta)

## 🚀 Step 5: Execute Model Training Pipeline

In [ ]:
from gwnet_trainer import train_full_gwnet

# Train Graph WaveNet on METR-LA
print('=== Starting Graph WaveNet Training on METR-LA ===')
best_la_checkpoint = train_full_gwnet(
    dataset_name='metr_la',
    num_epochs=15,
    batch_size=64,
    learning_rate=0.001,
    stride=2
)
print(f'[SUCCESS] Training Completed! Model saved to: {best_la_checkpoint}')

## 📊 Step 6: Model Forecast Evaluation & Visualization

In [ ]:
import matplotlib.pyplot as plt
from gwnet_adapter import UniversalPeMSAdapter

adapter = UniversalPeMSAdapter('metr_la')
dummy_input = np.random.uniform(20.0, 65.0, size=(13, 207))
preds = adapter.predict_next_15min(dummy_input)

plt.figure(figsize=(12, 4))
plt.plot(preds[0, :50], label='15-Min Predicted Sensor Speeds (mph)', color='#38bdf8')
plt.axhline(25.0, color='red', linestyle='--', label='Bottleneck Threshold (25 mph)')
plt.title('EquiTraffic-GPT Graph WaveNet 15-Minute Corridor Speed Forecasts')
plt.xlabel('Sensor ID (First 50 Nodes)')
plt.ylabel('Speed (mph)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()